# Local TTS Training
This notebook is configured to run on your local machine (Windows/Mac/Linux). It removes all Google Colab dependencies.

In [ ]:
# Cell 1: Setup Paths and Device
import os
import sys
import torch

# If running inside the 'notebooks' folder, move up to the main project folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cpu':
    print("⚠️ WARNING: Training on CPU will take a very long time. Consider using a small MAX_SAMPLES.")

In [ ]:
# Cell 2: Check & Preprocess Data
import glob
import numpy as np
from tqdm import tqdm
from src.preprocessing import audio_to_spectrogram
from src.config import Config

raw_wav_dir = os.path.join('data', 'raw', 'LJSpeech-1.1', 'wavs')
os.makedirs(Config.PROCESSED_DATA_DIR, exist_ok=True)

if not os.path.exists(raw_wav_dir):
    print(f"❌ ERROR: Could not find raw data at {raw_wav_dir}")
    print("Please download the LJSpeech dataset and extract it to data/raw/LJSpeech-1.1/")
else:
    wav_files = glob.glob(os.path.join(raw_wav_dir, '*.wav'))
    print(f"Found {len(wav_files)} audio files.")
    
    processed_files = glob.glob(os.path.join(Config.PROCESSED_DATA_DIR, '*.npy'))
    if len(processed_files) < len(wav_files):
        print("Preprocessing missing audio files...")
        for wav_file in tqdm(wav_files):
            base_name = os.path.basename(wav_file).replace('.wav', '.npy')
            out_path = os.path.join(Config.PROCESSED_DATA_DIR, base_name)
            if not os.path.exists(out_path):
                spec = audio_to_spectrogram(wav_file)
                np.save(out_path, spec)
    print("✅ Preprocessing complete!")

In [ ]:
# Cell 3: Training Configuration
MAX_SAMPLES = 500  # Set lower for local CPU testing!
BATCH_SIZE = 8
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
TEACHER_FORCING_RATIO = 0.5
TRAIN_SPLIT = 0.9

print("✅ Configuration set.")

In [ ]:
# Cell 4: Create DataLoaders
from torch.utils.data import DataLoader, random_split
from src.dataset import LJSpeechDataset, collate_fn

metadata_path = os.path.join('data', 'raw', 'LJSpeech-1.1', 'metadata.csv')
dataset = LJSpeechDataset(metadata_path=metadata_path, processed_dir=Config.PROCESSED_DATA_DIR)

if len(dataset) > MAX_SAMPLES:
    indices = list(range(MAX_SAMPLES))
    dataset = torch.utils.data.Subset(dataset, indices)

train_size = int(TRAIN_SPLIT * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"✅ Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

In [ ]:
# Cell 5: Initialize Model (With Resume Logic)
from src.model import Seq2SeqTTS
from src.preprocessing import get_vocab_size

vocab_size = get_vocab_size()
model = Seq2SeqTTS(vocab_size, Config.embed_dim, Config.hidden_dim, Config.spec_bins).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

CHECKPOINT_DIR = os.path.join('outputs', 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

if os.path.exists(checkpoint_path):
    print(f"Found existing model at {checkpoint_path}! Resuming training...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"✅ Resumed from Epoch {checkpoint.get('epoch', '?')}")
else:
    print("No previous checkpoint found. Starting fresh from Epoch 1.")
    
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {param_count:,}")

In [ ]:
# Cell 6: Training Loop
import json

def masked_mse_loss(prediction, target, lengths):
    batch_size, max_time, _ = target.shape
    time_indices = torch.arange(max_time, device=target.device).unsqueeze(0)
    mask = (time_indices < lengths.unsqueeze(1))
    frame_loss = torch.mean((prediction - target) ** 2, dim=2)
    frame_loss = frame_loss * mask.float()
    denominator = mask.float().sum().clamp(min=1.0)
    return frame_loss.sum() / denominator

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0
    
    for text, seq_lengths, target_spec, spec_lengths in train_loader:
        text, seq_lengths = text.to(device), seq_lengths.to(device)
        target_spec, spec_lengths = target_spec.to(device), spec_lengths.to(device)
        
        optimizer.zero_grad()
        prediction = model(text, seq_lengths, target_spec, TEACHER_FORCING_RATIO)
        loss = masked_mse_loss(prediction, target_spec, spec_lengths)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_train_loss += loss.item()
        
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for text, seq_lengths, target_spec, spec_lengths in val_loader:
            text, seq_lengths = text.to(device), seq_lengths.to(device)
            target_spec, spec_lengths = target_spec.to(device), spec_lengths.to(device)
            prediction = model(text, seq_lengths, target_spec, teacher_forcing_ratio=0.0)
            loss = masked_mse_loss(prediction, target_spec, spec_lengths)
            epoch_val_loss += loss.item()
            
    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train: {epoch_train_loss:.4f} | Val: {epoch_val_loss:.4f}")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss
        }, checkpoint_path)
        print("  -> Saved new best model.")

In [ ]:
# Cell 7: Inference
from src.preprocessing import text_to_sequence, spectrogram_to_audio
import soundfile as sf
import IPython.display as ipd

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

text = "This is running locally on my laptop."
print(f"Generating audio for: '{text}'")

with torch.no_grad():
    seq = torch.LongTensor(text_to_sequence(text)).unsqueeze(0).to(device)
    seq_lengths = torch.LongTensor([seq.size(1)]).to(device)
    dummy_target = torch.zeros(1, 300, Config.spec_bins).to(device)
    predicted_spec = model(seq, seq_lengths, dummy_target, teacher_forcing_ratio=0.0)
    
spec_np = predicted_spec.squeeze(0).cpu().numpy()
audio = spectrogram_to_audio(spec_np)

os.makedirs('outputs/samples', exist_ok=True)
out_wav = 'outputs/samples/local_notebook_output.wav'
sf.write(out_wav, audio, Config.sample_rate)
print(f"Saved to {out_wav}!")
ipd.Audio(out_wav)